In [2]:
import ccxt
import pandas as pd
import numpy as np
import time

# 1. CONFIGURACIÓN
exchange = ccxt.binance()
TIMEFRAME = '1h' 
PAIRS =['BTC/USDT', 'SOL/USDT']
# Desde Junio de 2021 (Para pillar el bull market de 2021, el crash de 2022 y la recuperación)
SINCE = exchange.parse8601('2021-06-01T00:00:00Z') 

def fetch_history(symbol, timeframe, since):
    print(f"\nDescargando {symbol} desde Junio 2021...")
    all_ohlcv =[]
    
    while since < exchange.milliseconds():
        try:
            ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since=since, limit=1000)
            if not ohlcv: break
            
            all_ohlcv.extend(ohlcv)
            since = ohlcv[-1][0] + 1 
            time.sleep(0.1) # Respetar límites de la API
            
            # Progreso
            last_date = pd.to_datetime(ohlcv[-1][0], unit='ms')
            print(f"   -> Datos bajados hasta: {last_date.strftime('%Y-%m-%d')}", end='\r')
            
        except Exception as e:
            print(f"\nError de conexión: {e}")
            time.sleep(5) # Si falla, esperar 5 seg y reintentar
            
    print(f"\n✅ {symbol} descarga completada.")
    df = pd.DataFrame(all_ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df.set_index('timestamp', inplace=True)
    return df[['close', 'volume']]

# 2. DESCARGA
btc = fetch_history('BTC/USDT', TIMEFRAME, SINCE)
sol = fetch_history('SOL/USDT', TIMEFRAME, SINCE)

# 3. FUSIÓN Y LIMPIEZA
print("\nAlineando datos de ambos activos...")
df = pd.concat([btc, sol], axis=1).dropna()
df.columns =['BTC', 'Vol_BTC', 'SOL', 'Vol_SOL']

# 4. FEATURE ENGINEERING (Indicadores)
print("Calculando Matemáticas e Indicadores Técnicos...")

# A. Retornos Logarítmicos
df['lret_btc'] = np.log(df['BTC'] / df['BTC'].shift(1))
df['lret_sol'] = np.log(df['SOL'] / df['SOL'].shift(1))

# B. Cambios de Volumen
df['vol_change_btc'] = np.log(df['Vol_BTC'] / df['Vol_BTC'].shift(1) + 1)
df['vol_change_sol'] = np.log(df['Vol_SOL'] / df['Vol_SOL'].shift(1) + 1)

# C. Ratio y Z-Score (Ventana 24 horas)
WINDOW = 24 
df['ratio'] = df['SOL'] / df['BTC']
df['ratio_mean'] = df['ratio'].rolling(window=WINDOW).mean()
df['ratio_std'] = df['ratio'].rolling(window=WINDOW).std()
df['z_score'] = (df['ratio'] - df['ratio_mean']) / df['ratio_std']

# D. RSI (Indicador de Fuerza)
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df['rsi'] = calculate_rsi(df['SOL'])

# E. Distancia a la Media Móvil (Indicador de Tendencia)
# Media móvil de 50 horas
df['sma_50'] = df['SOL'].rolling(window=50).mean()
df['dist_sma'] = (df['SOL'] - df['sma_50']) / df['sma_50']

# F. EL TARGET (Objetivo: Direccional en Dólares a 4 HORAS vista)
PREDICTION_HORIZON = 4  # <--- CAMBIO AQUÍ (Antes era 1)
df['future_sol'] = df['SOL'].shift(-PREDICTION_HORIZON)
df['Target'] = (df['future_sol'] > df['SOL']).astype(int)

# 5. LIMPIEZA FINAL DE INFINITOS Y NANS
df = df.replace([np.inf, -np.inf], np.nan).dropna()

print(f"\n📊 Dataset Final: {len(df)} horas de historia pura.")

# Guardar
csv_name = 'datos_crypto_sol_btc_1h_historico.csv'
df.to_csv(csv_name)
print(f"Guardado exitosamente en '{csv_name}'")


Descargando BTC/USDT desde Junio 2021...
   -> Datos bajados hasta: 2026-03-11
✅ BTC/USDT descarga completada.

Descargando SOL/USDT desde Junio 2021...
   -> Datos bajados hasta: 2026-03-11
✅ SOL/USDT descarga completada.

Alineando datos de ambos activos...
Calculando Matemáticas e Indicadores Técnicos...

📊 Dataset Final: 41815 horas de historia pura.
Guardado exitosamente en 'datos_crypto_sol_btc_1h_historico.csv'
